# Упражнение 2, част 2: подготовка и мащабиране на данните

**Машинно самообучение · зимен семестър 2026/2027**

**Repo:** https://github.com/mkorudzhiev/ML_26_27<br>
**Discord:** https://discord.gg/WKztMqX46

Мартин Коруджиев<br/>
mkorudzhiev@uni-sofia.bg<br/>
0885 71 31 60

## Съдържание

1. [Цел на упражнението](#cel)
2. [Файлове в упражнението](#failove)
3. [Подготовка и стартиране](#podgotovka)
4. [Част 1. Подготовка на числови и категорийни признаци](#chast-1)
5. [Част 2. Мащабиране](#chast-2)
6. [Част 3. Как мащабът променя сходството](#chast-3)
7. [Обобщение](#obobshtenie)
8. [Задача](#zadacha)

<a id="cel"></a>
## Цел на упражнението

В първата част разгледахме ирисите с pandas и обучихме готов модел
върху пълния набор Iris. Сега използваме учебния
`data/iris_with_missing.csv`. В него шест числови измервания и
шест стойности в категорийния признак `light_conditions` са изтрити
нарочно. Признакът за осветяване е измислен за упражнението.
Оригиналният Iris не съдържа такива сведения.

Ще отделим тестови редове, ще попълним пропуските, ще кодираме
категориите и ще мащабираме числовите признаци. После ще сравним
най-близките съседи и резултата на 3-NN при два начина за мащабиране.
Накрая ще приложим същите стъпки върху SAheart. Тетрадката се
изпълнява самостоятелно от чисто ядро.

<a id="failove"></a>
## Файлове в упражнението

Пътищата са относителни спрямо папката на тази тетрадка.

| Файл | За какво служи |
|---|---|
| `Упражнение_2_част_1.ipynb` | част 1: Pandas и първи стъпки със scikit-learn |
| `Упражнение_2_част_2.ipynb` | част 2: подготовка и мащабиране на данните |
| `ЗА_ВКЪЩИ_част_1.ipynb` | задачи за оглед и филтриране след част 1 |
| `ЗА_ВКЪЩИ_част_2.ipynb` | задачи за подготовка на данните след част 2 |
| `data/iris_with_missing.csv` | 150 ириса с четири измервания, измислени условия на осветяване и вид |
| `data/SAheart.data` | оригиналният набор SAheart от сайта на авторите |
| `data/saheart_exercise.csv` | учебно копие на SAheart за финалните задачи |
| `../environment.yml` | обща среда при работа с `conda` |
| `../requirements.txt` | общ списък с пакети при работа с `pip` |

Файлът е учебен вариант на Iris. Изтрити са шест числови измервания
и шест стойности на `light_conditions`. Останалите категории в
измислената колона са "слънце", "полусянка" и "сянка". Оригиналният
Iris не съдържа сведения за осветяване. Празната клетка означава
липсваща стойност, а не нула.

<a id="podgotovka"></a>
## Подготовка и стартиране

Използваме средата от първото упражнение. От папката на упражнение 2
можем да обновим пакетите след активиране на средата:

| Среда | Команди |
|---|---|
| Anaconda Prompt | `conda activate ml2627`<br/>`conda env update -f ../environment.yml` |
| PowerShell и `pip` | `..\.venv\Scripts\Activate.ps1`<br/>`python -m pip install -r ../requirements.txt` |

След това стартираме `jupyter lab` от тази папка, отваряме
`Упражнение_2_част_2.ipynb` и избираме ядрото "Python (МС 26/27)".

In [ ]:
import solution_2

%load_ext autoreload
%autoreload 2

import pandas as pd
from IPython.display import display

df = pd.read_csv("data/iris_with_missing.csv")
numeric_features = [
    "sepal_length_cm", "sepal_width_cm", "petal_length_cm", "petal_width_cm"
]
categorical_features = ["light_conditions"]
feature_names = numeric_features + categorical_features

print("Форма:", df.shape)
print("Пропуски в измерванията:", df[numeric_features].isna().sum().sum())
print("Пропуски в осветяването:", df[categorical_features].isna().sum().sum())
print("\nПропуски по колони:\n", df[feature_names].isna().sum(), sep='', end='\n\n')
display(df.head(3))

## Булеви маски

Булевата маска е масив от стойности `True` и `False`, който използваме, за да изберем само определени елементи от данните. Обикновено я създаваме чрез условие, например `age >= 50`.
Разликата е в това дали маската съдържа по една стойност за всеки ред, или по една стойност за всяка клетка.

### 1. Едномерна маска (вектор)

Едномерната маска съдържа по една булева стойност за всеки ред. Използваме я, когато искаме да изберем цели редове от таблицата.
В примера по-долу `mask` е `Series` с `True` за редовете, в които sepal_length_cm е >= 5, и `False` за останалите.
`df.loc[mask]` запазва само редовете, за които маската е `True`.

In [ ]:
mask = df["sepal_length_cm"] >= 5
print("Type:", type(mask))
print("Shape:", mask.shape)
display(mask.head())
display(df.loc[mask].head())

### 2. Двумерна маска (матрица)

Двумерната маска съдържа по една булева стойност за всяка клетка от таблицата.
Двумерната маска не означава непременно, че ще премахваме редове или колони.
В pandas тя често служи за маскиране на отделни стойности. 
Ако искаме от нея да получим маска за избор на редове или колони, можем да използваме `any()` или `all()`.
Важно е да използваме `axis=1`:

- `axis=0` - обобщаваме редовете и получаваме по една стойност за всяка колона.
- `axis=1` - обобщаваме колоните и получаваме по една стойност за всеки ред.

В примера по-долу `mask` е `DataFrame` със същите размери като `df`.
Всяка клетка е `True`, ако съответната стойност липсва, и `False` в противен случай.
Иползваме ~, за да обърнем маската - True ще означава, че редът няма пропуски.

**Важно**: `loc` работи само с едномерна маска!

In [ ]:
# За да махнем редовете с празни стойности:

# 1. създаваме маска, която е True, там където стойността липсва
mask = df.isna()
print("Type:", type(mask))
print("Shape:", mask.shape)
display(mask.iloc[10:15])

# 2. обръщаме маската и става True, само там където ИМАМЕ стойност
reversed_mask = ~mask
print("Type:", type(reversed_mask))
print("Shape:", reversed_mask.shape)
display(reversed_mask.iloc[10:15])

# 3. създаваме едномерна маска само с редовете, които за всяка колона имат стойност
vector_mask = reversed_mask.all(axis=1)
print("Type:", type(vector_mask))
print("Shape:", vector_mask.shape)
display(vector_mask.iloc[10:15])

# 4. създаваме под-таблица с loc
df_no_missing = df.loc[vector_mask]
print("Type:", type(df_no_missing))
print("Shape:", df_no_missing.shape)
display(df_no_missing.iloc[10:15])
print("Останали пропуски:", df_no_missing.isna().sum().sum())


# Всичко това може да се постигне в един ред ;)
df_no_missing_2 = df.loc[(~df.isna()).all(axis=1)]
print("Shape:", df_no_missing_2.shape)
print("Останали пропуски:", df_no_missing_2.isna().sum().sum())

In [ ]:
# --- ДА ОПИТАМЕ 1 ---
#
# 1. Нека използваме any() за постигане на същото - таблица без липсващи стойности
# 2. Нека извадим само редовете, които имат празни стойности

# solution_2.to_test_1(df)

<a id="chast-1"></a>
## Част 1. Подготовка на числови и категорийни признаци

Преди моделът да сравнява цветя, трябва да получи **числова** таблица без
пропуски. `SimpleImputer` научава стойности за попълване, а
`OneHotEncoder` превръща номинална категория в числови индикатори.

| Метод | Какво прави |
|---|---|
| `fit(train)` | научава правило само от тренировъчните редове |
| `transform(data)` | прилага същото правило към тренировъчни или тестови редове |
| `fit_transform(train)` | изпълнява двете стъпки върху тренировъчните редове |

Тестовите редове отделяме **преди** `fit`. Така стойностите, които
искаме да предскажем, не влияят на подготовката.

### 3. Първата стъпка винаги е намирането на редовете с празни данни

Празната клетка не означава нула. Първо виждаме редовете с
липсващо измерване или условие на осветяване.

In [ ]:
rows_with_missing = df.loc[df[feature_names].isna().any(axis=1)]
print("Редове с поне един пропуск:", len(rows_with_missing))
display(rows_with_missing)

### 4. Отделяне на признаците (по вид) и  на целта. Разделяне на тренировъчни и тестови редове

`X` съдържа четирите измервания и `light_conditions`, а `y` съдържа
вида `species`. В CSV файла редовете са подредени по вид.
`train_test_split` ги разбърква и разделя възпроизводимо.
`stratify=y` запазва дела на трите вида в двете части.
Разделението **трябва** да се направи преди попълването
на липсващите стойности!

In [ ]:
from sklearn.model_selection import train_test_split

X = df[feature_names]
y = df["species"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print("Тренировъчни:", X_train.shape, "тестови:", X_test.shape)
print("Числови пропуски, тренировъчни/тестови:",
      int(X_train[numeric_features].isna().sum().sum()),
      int(X_test[numeric_features].isna().sum().sum()))
print("Категорийни пропуски, тренировъчни/тестови:",
      int(X_train[categorical_features].isna().sum().sum()),
      int(X_test[categorical_features].isna().sum().sum()))

### 5. Попълване липсващите измервания

`SimpleImputer(strategy="median")` намира медианата на всяка числова
колона в тренировъчната част. Прилагаме четирите научени медиани и
към тестовите редове. Преобразувателят връща NumPy масив; създаваме
от него `DataFrame` със същите индекси и имена на колони.
`light_conditions` не участва в изчисляването на медианите.
Ще попълним неговите пропуски отделно.

| Стратегия | С какво попълва | Подходяща употреба |
|---|---|---|
| `"median"` | Медианата | Числови данни с крайни стойности |
| `"mean"` | Средното аритметично | Числови данни без силни крайни стойности |
| `"most_frequent"` | Най-често срещаната стойност | Често за категории като `light_conditions` |
| `"constant"` | Стойност, зададена чрез `fill_value` | Например категория `"неизвестно"` или число `0` |

In [ ]:
from sklearn.impute import SimpleImputer

imputer = SimpleImputer(strategy="median").fit(X_train[numeric_features])

X_train_filled = pd.DataFrame(
    imputer.transform(X_train[numeric_features]),
    index=X_train.index, columns=numeric_features
)

X_test_filled = pd.DataFrame(
    imputer.transform(X_test[numeric_features]),
    index=X_test.index, columns=numeric_features
)

print("Научени медиани:")
print(pd.Series(imputer.statistics_, index=numeric_features).round(2))
print("Останали без стойност:",
      int(X_train_filled.isna().sum().sum()),
      int(X_test_filled.isna().sum().sum()))

In [ ]:
# --- ДА ОПИТАМЕ 2 ---
# 1. Да намерим първия ТЕСТОВ ред с липсващо измерване.
# 2. Да покажем същия ред преди и след попълването чрез неговия индекс.
# 3. Коя колона е променена?

# solution_2.to_test_2(X_test[numeric_features], X_test_filled)

### 6. Попълване на категорийния признак

"слънце", "полусянка" и "сянка" са номинални категории.
(Номинална категория е категория, която обозначава принадлежност 
към определена група, като между категориите няма естествена подредба.)
Не можем да използваме медиана за текстови категории. С
`SimpleImputer(strategy="most_frequent")` научаваме най-честата
категория **само от тренировъчните редове** и попълваме пропуските
и в двете части. Празните клетки не участват в броенето.
Първо ще видим честотите в тренировъчната част ("слънце" трябва
да е най-честа), за да разберем избора на преобразувателя.
Тази замяна е просто допускане и увеличава броя на най-честата
категория. Ако искаме да запазим липсата като отделна информация,
можем вместо това да използваме `strategy="constant"` и нова
категория "неизвестно".

In [ ]:
from sklearn.preprocessing import OneHotEncoder

print("Категории в тренировъчните редове:")
print(X_train["light_conditions"].value_counts(dropna=False))
print()

category_imputer = SimpleImputer(strategy="most_frequent").fit(
    X_train[categorical_features]
)
X_train_cat_filled = pd.DataFrame(
    category_imputer.transform(X_train[categorical_features]),
    index=X_train.index, columns=categorical_features
)
X_test_cat_filled = pd.DataFrame(
    category_imputer.transform(X_test[categorical_features]),
    index=X_test.index, columns=categorical_features
)
print("Научена категория за попълване:", category_imputer.statistics_[0])
print("Останали категорийни пропуски:",
      int(X_train_cat_filled.isna().sum().sum()),
      int(X_test_cat_filled.isna().sum().sum()))

missing_category = X_train["light_conditions"].isna()
display(pd.DataFrame({
    "преди": X_train.loc[missing_category, "light_conditions"],
    "след": X_train_cat_filled.loc[missing_category, "light_conditions"]
}))

### 7. Кодиране на категорийния признак

След това `OneHotEncoder` създава по една индикаторна колона за всяка
от трите категории. Кодовете 1, 2 и 3 биха внушили ред и числови
разстояния, които не сме наблюдавали. `handle_unknown="ignore"`
позволява да преобразуваме и нова, непозната категория; тя получава
нули във всички научени индикаторни колони. Празните клетки обаче
попълваме **преди** кодирането, за да не бъдат третирани като
четвърта категория. С други думи OneHotEncoder заменя една 
категориална характеристика с няколко нови числови 
характеристики - по една за всяка категория.

In [ ]:
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False).fit(
    X_train_cat_filled
)
encoded_names = encoder.get_feature_names_out(categorical_features)
print("Имена на колони:", encoded_names)
X_train_encoded = pd.DataFrame(
    encoder.transform(X_train_cat_filled),
    index=X_train.index, columns=encoded_names
)
X_test_encoded = pd.DataFrame(
    encoder.transform(X_test_cat_filled),
    index=X_test.index, columns=encoded_names
)
print("Научени категории за кодиране:", encoder.categories_[0].tolist())
display(pd.concat([X_train_cat_filled.head(10),
                   X_train_encoded.head(10)], axis=1))

Параметърът `handle_unknown` на `OneHotEncoder` определя какво да се случи, 
когато при `transform()` се появи категория, която не е срещана при `fit()`.

Има четири възможни стойности:

| Стойност                | Поведение                                                                                                              |
| ----------------------- | ---------------------------------------------------------------------------------------------------------------------- |
| `"error"`               | Предизвиква грешка при непозната категория. Стойност по подразбиране.                                                  |
| `"ignore"`              | Кодира непознатата категория с нули във всички колони за съответната характеристика.                                   |
| `"infrequent_if_exist"` | Отнася непознатата категория към групата на редките категории, ако такава е създадена. Иначе се държи като `"ignore"`. |
| `"warn"`                | Извежда предупреждение, след което се държи като `"infrequent_if_exist"`.                 |

**Пример**

Да предположим, че енкодерът е обучен с две категории:

```
train = pd.DataFrame({"light_conditions": ["слънце", "сянка"]})
```

След това получаваме нова, непозната категория:

```
new = pd.DataFrame({"light_conditions": ["изкуствена светлина"]})
```

При `handle_unknown="error"`:

```
encoder = OneHotEncoder(handle_unknown="error", sparse_output=False)
encoder.fit(train)
encoder.transform(new)
# ValueError
```

При `handle_unknown="ignore"`:

```
encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
encoder.fit(train)
encoder.transform(new)
```

Резултат:

```
[[0. 0.]]
```

Получаваме две нули, защото непознатата категория не съответства на нито една от двете научени категории.

Когато имаме много категории, някои от които се срещат рядко, можем да ги обединим в обща група чрез `min_frequency`:

```
encoder = OneHotEncoder(handle_unknown="infrequent_if_exist", min_frequency=2, sparse_output=False)
```

Категориите, които се срещат по-малко от два пъти при обучението, ще бъдат обединени в една характеристика за редки категории.
Непознатите категории също ще се кодират чрез нея, ако тази група съществува.

Параметърът `sparse_output` на `OneHotEncoder` определя как ще се съхранява резултатът от преобразуването - 
като обикновен (плътен) масив или като разредена матрица.

Той не променя самото кодиране, а само начина, по който се представят получените нули и единици.

**Защо изобщо е необходим?**

При One-Hot Encoding всяка категория получава собствена колона. За даден пример обикновено само
една от тези колони има стойност `1`, а останалите са `0`.

Например:

```
слънце    → [0, 1, 0]
сянка     → [0, 0, 1]
полусянка → [1, 0, 0]
```

Ако имаме 1000 категории, за всеки пример ще получим 1000 стойности, от които 999 са нули.

Съхраняването на всички тези нули може да изисква много памет.

**Две възможности**

| Настройка             | Резултат                                                                                       |
| --------------------- | ---------------------------------------------------------------------------------------------- |
| `sparse_output=False` | Обикновен NumPy масив, който съхранява всички стойности, включително нулите.                   |
| `sparse_output=True`  | Разредена матрица (SciPy CSR), която съхранява основно ненулевите стойности и техните позиции. |

Стойността по подразбиране е `True`.

**Кога коя настройка е подходяща?**

- `False` - за учебни примери и малки набори от данни, когато искаме лесно да разглеждаме резултатите.
- `True` - за големи набори от данни с много категории, когато искаме да спестим памет.

**Важно**: `sparse_output=True` не означава, че премахваме нулите от данните или губим информация.
Нулите продължават да са част от логическото представяне на матрицата,
но не се съхраняват изрично една по една.

In [ ]:
# --- ДА ОПИТАМЕ 3 ---
#
# 1. Какво ще стане, ако кодираме суровата тренировъчна колона без предварително попълване?
# 2. Сравняваме категориите и броя one-hot колони с правилния encoder.

# solution_2.to_test_3(X_train, X_train_cat_filled, categorical_features)

In [ ]:
# --- ДА ОПИТАМЕ 4 ---
#
# "изкуствена светлина" не присъства в тренировъчните данни.
# Преобразуваме я със същия encoder. С какво резултатът се
# различава от кода на "слънце"? Какво означават нулите?

solution_2.to_test_4(encoder, categorical_features)

<a id="chast-2"></a>
## Част 2. Мащабиране

### 8. Защо мащабираме

Методите, основани на разстояние, сравняват числовите разлики между
примерите. По-голям диапазон в една колона може да има по-силен
принос към разстоянието. Четирите измервания на ирисите са в
сантиметри, но диапазоните им не са еднакви.

Ще сравним два начина за преобразуване на **всяка числова колона**:

| Метод | Правило | Тренировъчен резултат |
|---|---|---|
| Min-max | `(x - min) / (max - min)` | между 0 и 1 |
| Стандартизация | `(x - mean) / std` | средно 0 и стандартно отклонение 1 |

Минимумът, максимумът, средното и стандартното отклонение се научават
само от тренировъчните редове. `species` е цел и не се мащабира.
One-hot колоните остават 0 или 1.

### 9. Min-max мащабиране

`MinMaxScaler` научава минимума и максимума на попълнените
тренировъчни колони. За тестова стойност извън този диапазон
преобразуваният резултат може да е под 0 или над 1.

In [ ]:
from sklearn.preprocessing import MinMaxScaler

minmax_scaler = MinMaxScaler().fit(X_train_filled)
X_train_minmax = pd.DataFrame(
    minmax_scaler.transform(X_train_filled),
    index=X_train.index, columns=numeric_features
)
X_test_minmax = pd.DataFrame(
    minmax_scaler.transform(X_test_filled),
    index=X_test.index, columns=numeric_features
)
print("Тренировъчни минимуми:", X_train_minmax.min().round(2).to_dict())
print("Тренировъчни максимуми:", X_train_minmax.max().round(2).to_dict())
print("Най-малка тестова стойност:", round(X_test_minmax.min().min(), 3))
print("Най-голяма тестова стойност:", round(X_test_minmax.max().max(), 3))

In [ ]:
# --- ДА ОПИТАМЕ 5 ---
#
# 1. Проверяваме дали в РЕАЛНИТЕ тестови редове има стойности извън интервала [0, 1] след min-max мащабиране.
# 2. Отпечатваме общия им брой и колоните, в които се срещат, както и самите стойности.

# solution_2.to_test_5(X_test_minmax)

### 10. Стандартизация

`StandardScaler` научава средното и стандартното отклонение на всяка
попълнена тренировъчна колона. Използваме същите параметри за
тестовите редове. Тренировъчните колони ще имат средно близо до 0 и
стандартно отклонение близо до 1. За тестовите средното не е
задължително нула.

In [ ]:
from sklearn.preprocessing import StandardScaler

standard_scaler = StandardScaler().fit(X_train_filled)
X_train_scaled = pd.DataFrame(
    standard_scaler.transform(X_train_filled),
    index=X_train.index, columns=numeric_features
)
X_test_scaled = pd.DataFrame(
    standard_scaler.transform(X_test_filled),
    index=X_test.index, columns=numeric_features
)

print("Средни след стандартизация:", X_train_scaled.mean().round(2).to_dict())
print("Стандартни отклонения:", X_train_scaled.std(ddof=0).round(2).to_dict())
display(X_train_filled.head(3))
display(X_train_scaled.head(3).round(2))
print("Средни тестови след стандартизация:", X_test_scaled.mean().round(2).to_dict())
print("Тестови стандартни отклонения:", X_test_scaled.std(ddof=0).round(2).to_dict())

In [ ]:
# --- ДА ОПИТАМЕ 6 ---
#
# Намираме коя СТАНДАРТИЗИРАНА тестова колона има средно,
# най-далече от нулата. Отпечатваме името и средното.

# solution_2.to_test_6(X_test_scaled)

### 11. Събираме подготвените признаци

Събираме стандартизираните измервания и one-hot колоните в една
числова таблица. Индексите запазват съответствието между редовете
при `pd.concat`.

In [ ]:
X_train_ready = pd.concat([X_train_scaled, X_train_encoded], axis=1)
X_test_ready = pd.concat([X_test_scaled, X_test_encoded], axis=1)

print("Готови таблици:", X_train_ready.shape, X_test_ready.shape)
print("Останали пропуски:",
      int(X_train_ready.isna().sum().sum()),
      int(X_test_ready.isna().sum().sum()))
display(X_train_ready.head(3).round(2))

<a id="chast-3"></a>
## Част 3. Как мащабът променя сходството

### 12. Сравняваме най-близките съседи

Евклидовото разстояние между два числови реда събира разликите по
всички признаци: по-малко разстояние означава по-близки стойности.
За всеки тестов ирис намираме най-близкия тренировъчен ред след
min-max мащабиране и след стандартизация.

Тук сравняваме **само четирите измервания**, за да видим ефекта на
мащаба без измисления `light_conditions`. Разделянето на редовете и
попълнените стойности са едни и същи - променя се само мащабирането.

In [ ]:
import numpy as np
from sklearn.metrics import pairwise_distances

# Обединяваме тренировъчните и тестовите ириси.
X_all_minmax = pd.concat([X_train_minmax, X_test_minmax])
X_all_scaled = pd.concat([X_train_scaled, X_test_scaled])

# За всеки тестов ирис изчисляваме разстоянията до всички 150 ириса.
minmax_distances = pairwise_distances(X_test_minmax, X_all_minmax)
standard_distances = pairwise_distances(X_test_scaled, X_all_scaled)

# Изключваме разстоянието на всеки тестов ирис до самия него.
for i, idx in enumerate(X_test.index):
    j = X_all_minmax.index.get_loc(idx)
    minmax_distances[i, j] = np.inf
    standard_distances[i, j] = np.inf

# Намираме най-близкия съсед при двата метода.
neighbors = pd.DataFrame({
    "test_index": X_test.index,
    "minmax_neighbor": X_all_minmax.index[minmax_distances.argmin(axis=1)],
    "standard_neighbor": X_all_scaled.index[standard_distances.argmin(axis=1)]
})

neighbors["changed"] = (
    neighbors["minmax_neighbor"] != neighbors["standard_neighbor"]
)

print("Променен най-близък съсед:",
      int(neighbors["changed"].sum()),
      "от", len(neighbors))

display(neighbors.loc[neighbors["changed"]])

In [ ]:
# --- ДА ОПИТАМЕ 7 ---
#
# Избираме първия тестов ирис с променен съсед.
# Намираме вида на двата най-близки тренировъчни ириса чрез y_train.loc.
# Различни ли са видовете, или се е сменил само конкретният ред?

# solution_2.to_test_7(neighbors, y_train)

### 13. Проверяваме 3-NN при двата мащаба

Обучаваме по един `KNeighborsClassifier(n_neighbors=3)` върху
четирите числови измервания, мащабирани по двата начина. Използваме
едни и същи тренировъчни и тестови редове. Сравняваме точността
само за този избор на разделяне и параметри.

In [ ]:
from sklearn.metrics import accuracy_score
from sklearn.neighbors import KNeighborsClassifier

for label, train_data, test_data in [
    ("min-max", X_train_minmax, X_test_minmax),
    ("стандартизация", X_train_scaled, X_test_scaled),
]:
    model = KNeighborsClassifier(n_neighbors=3).fit(train_data, y_train)
    predictions = model.predict(test_data)
    print(label, "точност:", round(accuracy_score(y_test, predictions), 3))

Смяната на най-близкия ред показва, че мащабирането променя
разстоянието. Разликата в точността за тези 30 тестови цветя сама
по себе си не доказва, че единият метод винаги е по-добър.
Кодирането на категориен признак също променя представянето, но
измислените условия на осветяване не са реална характеристика на Iris.

<a id="obobshtenie"></a>
## Обобщение

- Отделяме тестовите редове преди да научим стойности за попълване,
  категории и мащаб.
- `fit` научава правило от тренировъчните редове, а `transform`
  прилага същото правило и към тестовите.
- Попълваме числовите пропуски с медиани, а категорийния признак
  с най-честата тренировъчна стойност, преди one-hot кодирането.
- Мащабът на числовите признаци влияе върху разстоянията и може да
  промени най-близките съседи и прогнозите.
- Проверяваме резултата на модел върху редове, които не е видял
  при обучението.

<a id="zadacha"></a>
## Задача

В първата част използвахме само `age` и `tobacco` от
`data/saheart_exercise.csv`, защото в тях няма пропуски. Сега ще
използваме **всичките осем числови признака** и категорийния
`famhist`. Ще попълним нарочно изтритите клетки, ще кодираме
категорията, ще стандартизираме числовите колони и ще обучим 5-NN.

`row.names` е идентификатор, а `chd` е целта. И двете остават извън
матрицата с признаци. Всички преобразуватели научават параметрите
си само от тренировъчните редове.

In [ ]:
heart = pd.read_csv("data/saheart_exercise.csv")
heart_X = heart.drop(columns=["row.names", "chd"])
heart_y = heart["chd"]
heart_numeric = heart_X.select_dtypes(include="number").columns.tolist()
heart_categorical = ["famhist"]

heart_X_train, heart_X_test, heart_y_train, heart_y_test = train_test_split(
    heart_X, heart_y, test_size=0.2, random_state=42, stratify=heart_y
)
print("Числови признаци:", heart_numeric)
print("Форми:", heart_X_train.shape, heart_X_test.shape)
print("Пропуски:", int(heart_X_train.isna().sum().sum()),
      int(heart_X_test.isna().sum().sum()))

### 1. Попълваме пропуските

Използваме `SimpleImputer(strategy="median")` за числовите колони и
`SimpleImputer(strategy="most_frequent")` за `famhist`. Научаваме
правилата само от `heart_X_train`. Създаваме четири таблици със
запазени индекси и имена на колони: `heart_train_num`,
`heart_test_num`, `heart_train_cat` и `heart_test_cat`.

In [ ]:
# heart_num_imputer = ...
# heart_cat_imputer = ...
# heart_train_num = ...
# heart_test_num = ...
# heart_train_cat = ...
# heart_test_cat = ...

### 2. Кодираме, стандартизираме и събираме

Обучаваме `OneHotEncoder` върху `heart_train_cat` и
`StandardScaler` върху `heart_train_num`. Прилагаме ги и към
тестовите таблици. Събираме резултатите по индекс в
`heart_train_ready` и `heart_test_ready`. Двете готови таблици
трябва да имат еднакви имена на колони и нито един пропуск.

In [ ]:
# heart_encoder = ...
# heart_scaler = ...
# heart_train_ready = ...
# heart_test_ready = ...
# print(...)

### 3. Обучаваме, проверяваме и прогнозираме

Обучаваме `KNeighborsClassifier(n_neighbors=5)` върху
`heart_train_ready` и намираме точността върху
`heart_test_ready`. Сравняваме я с
`DummyClassifier(strategy="most_frequent")`, обучен и проверен
върху същите редове.

Накрая избираме петия тестов ред чрез `iloc[[4]]`, предсказваме
`chd` и сравняваме с `heart_y_test.iloc[4]`. Какво показва
сравнението с базовия модел? Кои промени направихме спрямо
двупризнаковия модел от първата част? Можем ли да припишем
разликата в точността само на добавените признаци?

In [ ]:
# heart_model = ...
# heart_predictions = ...
# heart_accuracy = ...
# heart_baseline = ...
# baseline_accuracy = ...
# case = ...
# predicted_chd = ...
# actual_chd = ...
# print(...)